# Gold: lab_test_trends

Builds `adb_caresync.gold.lab_test_trends` (one row per test_name per calendar month) from silver lab_results and calendar, with a month-over-month LAG trend on abnormal rate, per STTM `sttm_silver_to_gold.xlsx` v1.0. Full rebuild via CREATE OR REPLACE TABLE.

In [0]:
%sql
-- Grain: one row per test_name per calendar month. Driving table: silver.lab_results grouped by test_name, year, month
-- Note: silver.lab_results contains exact duplicate lab_result_id rows; dedup to one row per lab test before aggregating
CREATE OR REPLACE TABLE adb_caresync.gold.lab_test_trends
COMMENT 'Gold lab test trends: one row per test type per month with abnormal/critical rates and month-over-month trend'
AS
WITH lab_results_dedup AS (
  SELECT *
  FROM adb_caresync.silver.lab_results
  QUALIFY row_number() OVER (PARTITION BY lab_result_id ORDER BY update_timestamp DESC, insert_timestamp DESC) = 1
),
monthly AS (
  SELECT
    test_name,
    year(test_date)                      AS year,
    month(test_date)                     AS month,
    count(*)                             AS total_tests,
    count_if(result_status = 'Abnormal') AS abnormal_test_count,
    count_if(result_status = 'Critical') AS critical_test_count
  FROM lab_results_dedup
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    m.*,
    CAST(m.abnormal_test_count * 100.0 / m.total_tests AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(m.critical_test_count * 100.0 / m.total_tests AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_lag AS (
  SELECT
    r.*,
    lag(r.abnormal_rate_pct) OVER (PARTITION BY r.test_name ORDER BY r.year, r.month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  w.test_name                                         AS test_name,
  CAST(w.year AS INT)                                 AS year,
  CAST(w.month AS INT)                                AS month,
  c.month_name                                        AS month_name,
  c.quarter_name                                      AS quarter_name,
  CAST(w.total_tests AS INT)                          AS total_tests,
  CAST(w.abnormal_test_count AS INT)                  AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                  AS critical_test_count,
  w.abnormal_rate_pct                                 AS abnormal_rate_pct,
  w.critical_rate_pct                                 AS critical_rate_pct,
  CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2)) AS prev_month_abnormal_rate_pct,
  CAST(w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_lag w
LEFT JOIN adb_caresync.silver.calendar c
  ON c.date_key = CAST(date_format(make_date(w.year, w.month, 1), 'yyyyMMdd') AS INT)

In [0]:
# Return the record count of the rebuilt gold table to the caller (e.g. job / dbutils.notebook.run)
updated_df = spark.table("adb_caresync.gold.lab_test_trends")
record_count = updated_df.count()
print(f"adb_caresync.gold.lab_test_trends record count: {record_count}")
dbutils.notebook.exit(str(record_count))